
# UNIVERSIDAD PRIVADA DOMINGO SAVIO
#
# **Serie: Machine Learning - Aprendizaje Supervisado**
#
# **Módulo 10: Pipeline Completo y Selección de Modelos**
#
# *Módulo Final Integrador*
#
# Docente: Ing. Franklin Mercado
#


## Prerrequisitos

Este módulo integra **todos los conceptos** de los módulos anteriores:
- Módulos 1-4: Fundamentos, regresión, escalado, regularización, CV
- Módulos 5-9: Clasificación, KNN, árboles, ensemble, SVM

---
## 1. El Flujo de Trabajo Completo de ML

### 1.1 Visión General

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    FLUJO DE TRABAJO PROFESIONAL DE ML                       │
└─────────────────────────────────────────────────────────────────────────────┘

┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  1. DATOS    │───►│ 2. EDA &     │───►│ 3. FEATURE   │───►│ 4. MODELADO  │
│  Recolección │    │ Limpieza     │    │ Engineering  │    │ & Selección  │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                                                                    │
                                                                    ▼
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│ 8. MONITOREO │◄───│ 7. DEPLOY    │◄───│ 6. MODELO    │◄───│ 5. TUNING    │
│ & Retraining │    │ Producción   │    │ FINAL        │    │ Hiperparams  │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

### 1.2 En este Módulo

Nos enfocaremos en los pasos **3, 4, 5 y 6** con un proyecto integrador completo.

---
## 2. Configuración del Entorno

In [ ]:
# ══════════════════════════════════════════════════════════════
# IMPORTACIÓN COMPLETA DE BIBLIOTECAS
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier
)
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    make_scorer
)

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Todas las bibliotecas importadas correctamente")
print(f"   Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---
## 3. Proyecto Integrador: Sistema de Predicción de Riesgo Cardiovascular

### 3.1 Contexto del Proyecto

El **Ministerio de Salud de Bolivia** nos ha encargado desarrollar un sistema de predicción de riesgo cardiovascular para los centros de salud primaria.

**Objetivo**: Identificar pacientes que requieren seguimiento especializado para prevenir eventos cardiovasculares.

**Requisitos**:
1. El modelo debe ser interpretable para los médicos
2. Debe priorizar **detectar pacientes de riesgo** (recall alto)
3. Debe funcionar con datos típicos de consulta de atención primaria
4. Documentación completa para implementación

In [ ]:

# GENERACIÓN DEL DATASET COMPLETO

def generar_dataset_cardiovascular(n_muestras=1500, random_state=42):
    """
    Genera un dataset realista de pacientes para predicción de riesgo cardiovascular.
    Simula datos de centros de salud primaria en Bolivia.
    """
    np.random.seed(random_state)
    
    # ─────────────────────────────────────────────────────────────
    # VARIABLES DEMOGRÁFICAS
    # ─────────────────────────────────────────────────────────────
    edad = np.random.normal(50, 15, n_muestras).clip(25, 85).astype(int)
    sexo = np.random.choice(['M', 'F'], n_muestras, p=[0.45, 0.55])
    
    departamento = np.random.choice(
        ['La Paz', 'Santa Cruz', 'Cochabamba', 'Oruro', 'Potosí', 
         'Tarija', 'Chuquisaca', 'Beni', 'Pando'],
        n_muestras,
        p=[0.25, 0.28, 0.18, 0.06, 0.05, 0.06, 0.05, 0.04, 0.03]
    )
    
    zona = np.random.choice(['Urbana', 'Rural'], n_muestras, p=[0.68, 0.32])
    
    # ─────────────────────────────────────────────────────────────
    # VARIABLES CLÍNICAS
    # ─────────────────────────────────────────────────────────────
    # Presión arterial (correlacionada con edad)
    presion_sistolica = 100 + 0.6 * edad + np.random.normal(0, 15, n_muestras)
    presion_diastolica = 60 + 0.35 * edad + np.random.normal(0, 10, n_muestras)
    
    # Glucosa en ayunas (mg/dL)
    glucosa = 80 + 0.5 * edad + np.random.normal(0, 25, n_muestras)
    glucosa = glucosa.clip(60, 300)
    
    # IMC
    imc = np.random.normal(27, 5, n_muestras).clip(16, 45)
    
    # Colesterol total (mg/dL)
    colesterol_total = 150 + 0.8 * edad + 2.5 * imc + np.random.normal(0, 35, n_muestras)
    
    # HDL (colesterol "bueno") - menor en hombres
    hdl_base = np.where(sexo == 'M', 42, 52)
    colesterol_hdl = hdl_base + np.random.normal(0, 12, n_muestras)
    colesterol_hdl = colesterol_hdl.clip(25, 90)
    
    # Triglicéridos
    trigliceridos = 100 + 1.5 * imc + np.random.normal(0, 50, n_muestras)
    trigliceridos = trigliceridos.clip(50, 500)
    
    # Frecuencia cardíaca en reposo
    frecuencia_cardiaca = 72 + np.random.normal(0, 12, n_muestras)
    frecuencia_cardiaca = frecuencia_cardiaca.clip(50, 110).astype(int)
    
    # ─────────────────────────────────────────────────────────────
    # FACTORES DE RIESGO
    # ─────────────────────────────────────────────────────────────
    fumador = np.random.choice(['No', 'Ex-fumador', 'Sí'], n_muestras, p=[0.65, 0.15, 0.20])
    
    consumo_alcohol = np.random.choice(
        ['Nunca', 'Ocasional', 'Regular', 'Frecuente'],
        n_muestras, p=[0.35, 0.40, 0.18, 0.07]
    )
    
    actividad_fisica = np.random.choice(
        ['Sedentario', 'Ligera', 'Moderada', 'Activo'],
        n_muestras, p=[0.30, 0.35, 0.25, 0.10]
    )
    
    antecedentes_familiares = np.random.choice([0, 1], n_muestras, p=[0.60, 0.40])
    diabetes_diagnosticada = np.random.choice([0, 1], n_muestras, p=[0.88, 0.12])
    hipertension_diagnosticada = np.random.choice([0, 1], n_muestras, p=[0.75, 0.25])
    
    # ─────────────────────────────────────────────────────────────
    # VARIABLE TARGET: RIESGO CARDIOVASCULAR
    # ─────────────────────────────────────────────────────────────
    # Calcular score de riesgo basado en factores conocidos
    riesgo_score = (
        0.04 * (edad - 45) +
        0.03 * (presion_sistolica - 120) +
        0.025 * (glucosa - 100) +
        0.08 * (imc - 25) +
        0.02 * (colesterol_total - 200) +
        -0.03 * (colesterol_hdl - 50) +
        0.015 * (trigliceridos - 150) +
        np.where(sexo == 'M', 0.5, 0) +
        np.where(fumador == 'Sí', 1.8, np.where(fumador == 'Ex-fumador', 0.5, 0)) +
        np.where(actividad_fisica == 'Sedentario', 1.0, 
                 np.where(actividad_fisica == 'Ligera', 0.3, 
                          np.where(actividad_fisica == 'Moderada', -0.3, -0.8))) +
        1.5 * antecedentes_familiares +
        2.0 * diabetes_diagnosticada +
        1.2 * hipertension_diagnosticada +
        np.random.normal(0, 1.5, n_muestras)
    )
    
    # Convertir a probabilidad
    prob_riesgo = 1 / (1 + np.exp(-(riesgo_score - 3)))
    riesgo_alto = (np.random.random(n_muestras) < prob_riesgo).astype(int)
    
    # ─────────────────────────────────────────────────────────────
    # CREAR DATAFRAME
    # ─────────────────────────────────────────────────────────────
    df = pd.DataFrame({
        'edad': edad,
        'sexo': sexo,
        'departamento': departamento,
        'zona': zona,
        'presion_sistolica': presion_sistolica.round(1),
        'presion_diastolica': presion_diastolica.round(1),
        'glucosa': glucosa.round(1),
        'imc': imc.round(2),
        'colesterol_total': colesterol_total.round(1),
        'colesterol_hdl': colesterol_hdl.round(1),
        'trigliceridos': trigliceridos.round(1),
        'frecuencia_cardiaca': frecuencia_cardiaca,
        'fumador': fumador,
        'consumo_alcohol': consumo_alcohol,
        'actividad_fisica': actividad_fisica,
        'antecedentes_familiares': antecedentes_familiares,
        'diabetes': diabetes_diagnosticada,
        'hipertension': hipertension_diagnosticada,
        'riesgo_cardiovascular': riesgo_alto
    })
    
    # Introducir algunos valores faltantes (realismo)
    np.random.seed(random_state + 1)
    for col in ['colesterol_hdl', 'trigliceridos', 'glucosa']:
        mask = np.random.random(n_muestras) < 0.03  # 3% missing
        df.loc[mask, col] = np.nan
    
    return df

# Generar dataset
df = generar_dataset_cardiovascular(n_muestras=1500, random_state=RANDOM_STATE)

print("═" * 70)
print("DATASET: PREDICCIÓN DE RIESGO CARDIOVASCULAR")
print("═" * 70)
print(f"\n  Dimensiones: {df.shape}")
print(f"  Registros: {len(df):,}")
print(f"  Features: {len(df.columns) - 1}")
df.head()

---
## 4. Análisis Exploratorio de Datos (EDA)

In [ ]:

# INFORMACIÓN GENERAL DEL DATASET


print("═" * 70)
print("INFORMACIÓN DEL DATASET")
print("═" * 70)

print("\n📊 Tipos de datos:")
print(df.dtypes.to_string())

print("\n📊 Valores faltantes:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, 'Porcentaje': missing_pct})
print(missing_df[missing_df['Faltantes'] > 0].to_string())

In [ ]:

# DISTRIBUCIÓN DE LA VARIABLE TARGET


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribución general
target_counts = df['riesgo_cardiovascular'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Bajo Riesgo (0)', 'Alto Riesgo (1)'], target_counts.values, 
            color=colors, edgecolor='black')
for i, (count, pct) in enumerate(zip(target_counts.values, target_counts.values/len(df)*100)):
    axes[0].text(i, count + 20, f'{count}\n({pct:.1f}%)', ha='center', fontsize=12)
axes[0].set_ylabel('Número de Pacientes')
axes[0].set_title('Distribución de Riesgo Cardiovascular', fontsize=14, fontweight='bold')

# Por departamento
risk_by_dept = df.groupby('departamento')['riesgo_cardiovascular'].mean().sort_values(ascending=True)
axes[1].barh(risk_by_dept.index, risk_by_dept.values * 100, color='steelblue', edgecolor='black')
axes[1].set_xlabel('% de Pacientes con Alto Riesgo')
axes[1].set_title('Riesgo por Departamento', fontsize=14, fontweight='bold')
axes[1].axvline(x=df['riesgo_cardiovascular'].mean()*100, color='red', linestyle='--', 
                label=f'Promedio Nacional ({df["riesgo_cardiovascular"].mean()*100:.1f}%)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📊 Balance de clases: {target_counts[0]/target_counts[1]:.2f}:1 (Bajo:Alto)")

In [ ]:

# ANÁLISIS DE VARIABLES NUMÉRICAS


num_cols = ['edad', 'presion_sistolica', 'presion_diastolica', 'glucosa', 
            'imc', 'colesterol_total', 'colesterol_hdl', 'trigliceridos', 
            'frecuencia_cardiaca']

fig, axes = plt.subplots(3, 3, figsize=(14, 12))

for ax, col in zip(axes.flatten(), num_cols):
    for riesgo, color, label in [(0, '#2ecc71', 'Bajo Riesgo'), (1, '#e74c3c', 'Alto Riesgo')]:
        data = df[df['riesgo_cardiovascular'] == riesgo][col].dropna()
        ax.hist(data, bins=25, alpha=0.6, color=color, label=label, edgecolor='black')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)
    ax.set_title(f'Distribución: {col}', fontsize=10, fontweight='bold')

plt.suptitle('Variables Numéricas por Nivel de Riesgo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Preprocesamiento con ColumnTransformer

### 5.1 Identificar Tipos de Features

In [ ]:

# CLASIFICAR FEATURES POR TIPO


# Separar features y target
X = df.drop('riesgo_cardiovascular', axis=1)
y = df['riesgo_cardiovascular']

# Identificar tipos de columnas
num_features = ['edad', 'presion_sistolica', 'presion_diastolica', 'glucosa',
                'imc', 'colesterol_total', 'colesterol_hdl', 'trigliceridos',
                'frecuencia_cardiaca']

cat_features_nominal = ['sexo', 'departamento', 'zona']

cat_features_ordinal = ['fumador', 'consumo_alcohol', 'actividad_fisica']

bin_features = ['antecedentes_familiares', 'diabetes', 'hipertension']

print("═" * 70)
print("CLASIFICACIÓN DE FEATURES")
print("═" * 70)
print(f"\n  Numéricas ({len(num_features)}): {num_features}")
print(f"\n  Categóricas Nominales ({len(cat_features_nominal)}): {cat_features_nominal}")
print(f"\n  Categóricas Ordinales ({len(cat_features_ordinal)}): {cat_features_ordinal}")
print(f"\n  Binarias ({len(bin_features)}): {bin_features}")

In [ ]:

# CREAR PIPELINE DE PREPROCESAMIENTO


# Pipeline para features numéricas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Manejar NaN
    ('scaler', StandardScaler())                    # Escalar
])

# Pipeline para categóricas nominales (One-Hot)
cat_nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Pipeline para categóricas ordinales
cat_ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[
        ['No', 'Ex-fumador', 'Sí'],                    # fumador
        ['Nunca', 'Ocasional', 'Regular', 'Frecuente'], # consumo_alcohol
        ['Sedentario', 'Ligera', 'Moderada', 'Activo']  # actividad_fisica
    ]))
])

# Combinar todo con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat_nom', cat_nominal_pipeline, cat_features_nominal),
        ('cat_ord', cat_ordinal_pipeline, cat_features_ordinal),
        ('bin', 'passthrough', bin_features)  # Ya son binarias, no procesar
    ],
    remainder='drop'
)

print("✅ Preprocessor creado exitosamente")

In [ ]:

# DIVISIÓN TRAIN/TEST


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train)} muestras")
print(f"Test:  {len(X_test)} muestras")
print(f"\nProporción riesgo alto - Train: {y_train.mean():.3f}")
print(f"Proporción riesgo alto - Test:  {y_test.mean():.3f}")

---
## 6. Comparación Sistemática de Modelos

In [ ]:

# DEFINIR MODELOS A COMPARAR


modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=7, 
                                             random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                     random_state=RANDOM_STATE),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
}

print(f"✅ {len(modelos)} modelos definidos para comparación")

In [ ]:

# EVALUACIÓN CON MÚLTIPLES MÉTRICAS


scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

resultados = []

print("═" * 80)
print("COMPARACIÓN DE MODELOS (5-Fold Cross-Validation)")
print("═" * 80)
print(f"\n  {'Modelo':<22} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC-ROC':>10}")
print(f"  {'-'*22} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for nombre, modelo in modelos.items():
    # Crear pipeline completo
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])
    
    # Cross-validation con múltiples métricas
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    resultado = {
        'Modelo': nombre,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'AUC-ROC': scores['test_roc_auc'].mean(),
        'Accuracy_std': scores['test_accuracy'].std(),
        'AUC_std': scores['test_roc_auc'].std()
    }
    resultados.append(resultado)
    
    print(f"  {nombre:<22} {resultado['Accuracy']:>10.4f} {resultado['Precision']:>10.4f} "
          f"{resultado['Recall']:>10.4f} {resultado['F1']:>10.4f} {resultado['AUC-ROC']:>10.4f}")

resultados_df = pd.DataFrame(resultados)

In [ ]:

# VISUALIZACIÓN DE RESULTADOS


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Todas las métricas
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
x = np.arange(len(resultados_df))
width = 0.15

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, resultados_df[metric], width, label=metric)

axes[0].set_xticks(x + width*2)
axes[0].set_xticklabels(resultados_df['Modelo'], rotation=45, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Comparación de Métricas por Modelo', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0.5, 1.0)

# Gráfico 2: AUC-ROC con barras de error
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(resultados_df)))
sorted_df = resultados_df.sort_values('AUC-ROC', ascending=True)

axes[1].barh(sorted_df['Modelo'], sorted_df['AUC-ROC'], 
             xerr=sorted_df['AUC_std']*2, color=colors, edgecolor='black', capsize=4)
axes[1].set_xlabel('AUC-ROC (± 2 std)')
axes[1].set_title('Ranking de Modelos por AUC-ROC', fontsize=14, fontweight='bold')
axes[1].set_xlim(0.6, 1.0)

plt.tight_layout()
plt.show()

# Mejor modelo por métrica
print("\n📊 MEJOR MODELO POR MÉTRICA:")
for metric in metrics:
    best_idx = resultados_df[metric].idxmax()
    print(f"   {metric:<12}: {resultados_df.loc[best_idx, 'Modelo']} ({resultados_df.loc[best_idx, metric]:.4f})")

---
## 7. Optimización del Mejor Modelo

In [ ]:

# SELECCIÓN Y TUNING DEL MEJOR MODELO


# Basándonos en los resultados, optimizamos Random Forest y Gradient Boosting

# Pipeline base
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

# Grid de hiperparámetros
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

# GridSearchCV optimizando para RECALL (importante en contexto médico)
grid_search = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=5,
    scoring='recall',  # Priorizamos detectar pacientes de riesgo
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

print("Optimizando Random Forest para RECALL...")
grid_search.fit(X_train, y_train)

print("\n═" * 70)
print("MEJORES HIPERPARÁMETROS (Optimizado para Recall)")
print("═" * 70)
print(f"\n  Mejor Recall (CV): {grid_search.best_score_:.4f}")
print(f"\n  Parámetros óptimos:")
for param, value in grid_search.best_params_.items():
    print(f"    {param.replace('classifier__', '')}: {value}")

---
## 8. Evaluación Final en Conjunto de Test

In [ ]:

# MODELO FINAL: EVALUACIÓN EN TEST


modelo_final = grid_search.best_estimator_

# Predicciones
y_pred = modelo_final.predict(X_test)
y_proba = modelo_final.predict_proba(X_test)[:, 1]

print("═" * 70)
print("EVALUACIÓN FINAL EN CONJUNTO DE TEST")
print("═" * 70)

print(f"\n  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}  ← Objetivo principal")
print(f"  F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\n" + "─" * 70)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("─" * 70)
print(classification_report(y_test, y_pred, 
                            target_names=['Bajo Riesgo', 'Alto Riesgo']))

In [ ]:

# MATRIZ DE CONFUSIÓN Y CURVA ROC


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Bajo', 'Pred: Alto'],
            yticklabels=['Real: Bajo', 'Real: Alto'],
            annot_kws={'size': 16})
axes[0].set_title('Matriz de Confusión', fontsize=14, fontweight='bold')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'ROC (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Random (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.2)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].set_xlim(-0.02, 1.02)
axes[1].set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

# Interpretación de la matriz de confusión
TN, FP, FN, TP = cm.ravel()
print("\n📊 INTERPRETACIÓN CLÍNICA:")
print(f"   • Pacientes de bajo riesgo correctamente identificados: {TN}")
print(f"   • Pacientes de alto riesgo correctamente identificados: {TP}")
print(f"   • Falsas alarmas (bajo riesgo marcados como alto): {FP}")
print(f"   • Pacientes de riesgo NO detectados (peligroso): {FN} ⚠️")

---
## 9. Importancia de Features

In [ ]:

# IMPORTANCIA DE FEATURES DEL MODELO FINAL


# Obtener nombres de features después del preprocesamiento
feature_names = (
    num_features +
    list(modelo_final.named_steps['preprocessor']
         .named_transformers_['cat_nom']
         .named_steps['onehot']
         .get_feature_names_out(cat_features_nominal)) +
    cat_features_ordinal +
    bin_features
)

# Importancias
importances = modelo_final.named_steps['classifier'].feature_importances_

# Crear DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importancia': importances
}).sort_values('Importancia', ascending=True)

# Top 15
top_features = importance_df.tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(top_features['Importancia'] / top_features['Importancia'].max())

ax.barh(top_features['Feature'], top_features['Importancia'], color=colors, edgecolor='black')
ax.set_xlabel('Importancia', fontsize=12)
ax.set_title('Top 15 Features más Importantes para Predecir Riesgo Cardiovascular',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 TOP 5 FEATURES:")
for i, row in importance_df.tail(5).iloc[::-1].iterrows():
    print(f"   {row['Feature']}: {row['Importancia']:.4f}")

---
## 10. Documentación y Reporte Final

In [ ]:
# ══════════════════════════════════════════════════════════════
# REPORTE EJECUTIVO FINAL
# ══════════════════════════════════════════════════════════════

print("═" * 80)
print("              REPORTE EJECUTIVO: SISTEMA DE PREDICCIÓN DE RIESGO")
print("                           CARDIOVASCULAR - BOLIVIA")
print("═" * 80)

print("""
📋 RESUMEN DEL PROYECTO
─────────────────────────────────────────────────────────────────────────────

  OBJETIVO: Desarrollar un modelo de ML para identificar pacientes con alto
            riesgo cardiovascular en centros de salud primaria de Bolivia.

  DATOS:    1,500 registros de pacientes con 18 variables clínicas y
            demográficas de los 9 departamentos de Bolivia.

─────────────────────────────────────────────────────────────────────────────
""")

print(f"""
🔬 METODOLOGÍA
─────────────────────────────────────────────────────────────────────────────

  1. Preprocesamiento diferenciado:
     • Variables numéricas: Imputación (mediana) + Estandarización
     • Variables categóricas nominales: One-Hot Encoding
     • Variables categóricas ordinales: Ordinal Encoding

  2. Modelos evaluados (5-Fold CV):
     • Logistic Regression, KNN, Decision Tree
     • Random Forest, Gradient Boosting, SVM

  3. Optimización:
     • GridSearchCV para Random Forest
     • Métrica objetivo: RECALL (detectar pacientes de riesgo)

─────────────────────────────────────────────────────────────────────────────
""")

print(f"""
📊 RESULTADOS DEL MODELO FINAL (Random Forest Optimizado)
─────────────────────────────────────────────────────────────────────────────

  Métricas en conjunto de TEST (n={len(y_test)}):

    • Accuracy:  {accuracy_score(y_test, y_pred):.1%}
    • Precision: {precision_score(y_test, y_pred):.1%}
    • RECALL:    {recall_score(y_test, y_pred):.1%}  ← Métrica principal
    • F1-Score:  {f1_score(y_test, y_pred):.1%}
    • AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}

─────────────────────────────────────────────────────────────────────────────
""")

print(f"""
🏥 INTERPRETACIÓN CLÍNICA
─────────────────────────────────────────────────────────────────────────────

  De cada 100 pacientes con ALTO RIESGO real:
    • El modelo identifica correctamente a {recall_score(y_test, y_pred)*100:.0f} de ellos
    • {(1-recall_score(y_test, y_pred))*100:.0f} pacientes de riesgo NO serían detectados

  De cada 100 pacientes marcados como ALTO RIESGO por el modelo:
    • {precision_score(y_test, y_pred)*100:.0f} realmente tienen alto riesgo
    • {(1-precision_score(y_test, y_pred))*100:.0f} son falsas alarmas

  TOP 5 FACTORES DE RIESGO IDENTIFICADOS:
""")
for i, row in importance_df.tail(5).iloc[::-1].iterrows():
    print(f"    {i+1}. {row['Feature']}")

print(f"""
─────────────────────────────────────────────────────────────────────────────

✅ RECOMENDACIONES
─────────────────────────────────────────────────────────────────────────────

  1. Implementar el modelo como herramienta de apoyo a la decisión médica
  2. Los pacientes marcados como "Alto Riesgo" deben ser evaluados por
     un especialista
  3. Monitorear el rendimiento del modelo cada 6 meses y reentrenar si
     la precisión cae por debajo del 75%
  4. Priorizar la recolección de datos de colesterol HDL y triglicéridos
     (actualmente con ~3% de valores faltantes)

═════════════════════════════════════════════════════════════════════════════
  Elaborado por: Sistema ML - UCB
  Fecha: {datetime.now().strftime('%Y-%m-%d')}
═════════════════════════════════════════════════════════════════════════════
""")

---
## 11. Resumen del Módulo y de la Serie Completa

### 11.1 Lo que Aprendimos en este Módulo

1. **ColumnTransformer**: Preprocesamiento diferenciado por tipo de feature
2. **Comparación sistemática**: Evaluar múltiples modelos con las mismas condiciones
3. **Múltiples métricas**: No confiar en una sola métrica
4. **GridSearchCV**: Optimización eficiente de hiperparámetros
5. **Documentación**: Comunicar resultados de forma profesional

### 11.2 Resumen de la Serie Completa

| Módulo | Tema | Conceptos Clave |
|--------|------|----------------|
| 1 | Introducción y Flujo | Tipos de ML, train/test split |
| 2 | Regresión Lineal Simple | OLS, MSE, R² |
| 3 | Regresión Múltiple | Feature scaling, data leakage |
| 4 | Regularización | Ridge, Lasso, CV |
| 5 | Clasificación | Logística, métricas de clasificación |
| 6 | KNN | Distancias, elección de K |
| 7 | Árboles | Gini, entropía, poda |
| 8 | Ensemble | Bagging, Random Forest |
| 9 | SVM | Márgenes, kernels |
| 10 | Pipeline Completo | Integración, proyecto profesional |

---
## 12. Ejercicio Final Integrador

### 🎯 Proyecto: Sistema de Predicción Alternativo

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎯 EJERCICIO FINAL: VOTING CLASSIFIER
# ═══════════════════════════════════════════════════════════════

# Tu tarea:
# 1. Crea un VotingClassifier que combine los 3 mejores modelos
# 2. Usa voting='soft' para promediar probabilidades
# 3. Compara el rendimiento con el Random Forest individual
# 4. ¿El ensemble mejora el rendimiento?

# Tu código aquí
